# Customer Segmentation Using Clustering

This notebook demonstrates the full ML pipeline for customer segmentation.
It mirrors what the web application does, but in an interactive format.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
print('Libraries loaded successfully')

## 1. Load and Explore Data

In [ ]:
df = pd.read_csv('../data/sample_customer_data.csv')
print(f'Shape: {df.shape}')
print(f'Missing values:\n{df.isnull().sum()}')
df.head()

In [ ]:
df.describe()

## 2. Data Preprocessing

In [ ]:
# Remove duplicates
print(f'Duplicates: {df.duplicated().sum()}')
df = df.drop_duplicates()

# Handle missing values
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Encode Gender
df['Gender_encoded'] = (df['Gender'] == 'Male').astype(int)

print('Preprocessing complete')
print(f'Records after cleaning: {len(df)}')

## 3. Feature Selection & Scaling

In [ ]:
features = ['Age', 'AnnualIncome', 'SpendingScore', 'PurchaseFrequency']
X = df[features].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f'Feature matrix shape: {X_scaled.shape}')

## 4. Elbow Method

In [ ]:
wcss = []
sil_scores = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(k_range, wcss, 'b-o', linewidth=2)
ax1.set_title('Elbow Method — WCSS vs K')
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('WCSS')
ax1.grid(True, alpha=0.3)

ax2.plot(k_range, sil_scores, 'g-o', linewidth=2)
ax2.set_title('Silhouette Score vs K')
ax2.set_xlabel('Number of Clusters (K)')
ax2.set_ylabel('Silhouette Score')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/charts/elbow_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Optimal K (Silhouette): {list(k_range)[sil_scores.index(max(sil_scores))]}')

## 5. K-Means Clustering

In [ ]:
optimal_k = 5
km = KMeans(n_clusters=optimal_k, n_init=10, random_state=42)
km_labels = km.fit_predict(X_scaled)

print(f'K-Means Results (K={optimal_k}):')
print(f'  WCSS:             {km.inertia_:.2f}')
print(f'  Silhouette Score: {silhouette_score(X_scaled, km_labels):.4f}')
print(f'  Davies-Bouldin:   {davies_bouldin_score(X_scaled, km_labels):.4f}')

## 6. Hierarchical Clustering

In [ ]:
hc = AgglomerativeClustering(n_clusters=optimal_k, linkage='ward')
hc_labels = hc.fit_predict(X_scaled)

print(f'Hierarchical Results (K={optimal_k}):')
print(f'  Silhouette Score: {silhouette_score(X_scaled, hc_labels):.4f}')
print(f'  Davies-Bouldin:   {davies_bouldin_score(X_scaled, hc_labels):.4f}')

## 7. DBSCAN

In [ ]:
db = DBSCAN(eps=0.5, min_samples=5)
db_labels = db.fit_predict(X_scaled)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
noise = (db_labels == -1).sum()
print(f'DBSCAN Results:')
print(f'  Clusters: {n_clusters}, Noise points: {noise}')
if n_clusters >= 2:
    mask = db_labels != -1
    print(f'  Silhouette (no noise): {silhouette_score(X_scaled[mask], db_labels[mask]):.4f}')

## 8. PCA Visualization

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#6366f1', '#10b981', '#f59e0b', '#ef4444', '#8b5cf6']

for k in range(optimal_k):
    mask = km_labels == k
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], c=colors[k % len(colors)], label=f'Cluster {k}', alpha=0.6)
axes[0].set_title('K-Means Clusters (PCA)')
axes[0].legend()
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')

for k in range(optimal_k):
    mask = hc_labels == k
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], c=colors[k % len(colors)], label=f'Cluster {k}', alpha=0.6)
axes[1].set_title('Hierarchical Clusters (PCA)')
axes[1].legend()
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')

plt.tight_layout()
plt.savefig('../results/charts/pca_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Cluster Profiles

In [ ]:
df_result = df[features].copy()
df_result['Cluster'] = km_labels

profile = df_result.groupby('Cluster')[features].mean().round(2)
profile['Count'] = df_result.groupby('Cluster').size()
profile['Pct'] = (profile['Count'] / len(df_result) * 100).round(1)
print('Cluster Profiles:')
print(profile.to_string())